# assignment step 1:  movie review sentiment classifier

`Jacob Poirier`

Labels: `0 = negative`, `1 = positive`.

The training set contains 240 reviews (180 positive, 60 negative), and the public test set contains 400 reviews (200 positive,200 negative).

## chosen model structure

I used a CountVectorizer + Logistic Regression

review → word count features → logistic regression → 0/1

CountVectorizer converts each new review into word occurrence features. I also used word unigrams because the training set thats given is very small, this keeps the feature space more simple and reduces overfitting.The logistic regression uses L2 regularization and balanced class weights

In [ ]:
%pip install numpy pandas matplotlib scikit-learn joblib

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
train_df = pd.read_csv("train.csv")
public_test_df = pd.read_csv("public_test.csv")

print("training shape:", train_df.shape)
print("public testing shape:", public_test_df.shape)
print("\ntraining labels:")
print(train_df["label"].value_counts())
print("\npublic testing labels:")
print(public_test_df["label"].value_counts())

## how to handle the small training set with the imbalanced types of reviews 

The training data is small and has a 3:1 positive-to-negative imbalance.

- I used a simple linear classifier to reduce overfitting, smaller training sets have this common issue.
- L2 regularization controls the magnitude of model coefficients
- class_weight="balanced" gives some more weight to the minority negative class so that they aren't underestimated by the model.
- a stratified validation split preserves the class proportions despite the imbalance.
- the final model is retrained using all 240 labeled reviews after the model configuration is selected.

In [ ]:
X = train_df["text"].fillna("").astype(str)
y = train_df["label"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("training samples:", len(X_train))
print("validation samples:", len(X_val))
print("training distribution:\n", y_train.value_counts())
print("validation distribution:\n", y_val.value_counts())

## training techniques

- **Feature representation:** word-count unigrams (`ngram_range=(1,1)`).
- **Regularization:** L2, with `C=1.0`.
- **Class imbalance:** `class_weight="balanced"`.
- **Solver/optimizer:** `liblinear`, suitable for a small binary classification problem.
- **Learning rate:** no manual learning rate was used because `liblinear` is a conventional logistic regression solver rather than a mini batch gradient descent implementation.
- **Batch size:** wasn't really applicable for this type of solver
- **Maximum iterations:** 1000.
- **Random state:** 42 for reproducibility.

In [ ]:
def make_model():
    return Pipeline([
        ("vectorizer", CountVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 1),
            min_df=1,
            max_features=20000
        )),
        ("classifier", LogisticRegression(
            C=1.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        ))
    ])

model = make_model()
model.fit(X_train, y_train)

val_predictions = model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)
val_cm = confusion_matrix(y_val, val_predictions)

print("validation accuracy=", val_accuracy)
print("validation confusion matrix=")
print(val_cm)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=val_cm,
    display_labels=["Negative", "Positive"]
).plot()
plt.title("validation confusion matrix")
plt.show()

## final model structure

after selecting the model structure, the final model is trained on all of the 240 training reviews. This gives the classifier access to every labeled training example before using the public test evaluation.

In [ ]:
final_model = make_model()
final_model.fit(X, y)

X_public_test = public_test_df["text"].fillna("").astype(str)
y_public_test = public_test_df["label"].astype(int)

public_predictions = final_model.predict(X_public_test)
public_accuracy = accuracy_score(y_public_test, public_predictions)
public_cm = confusion_matrix(y_public_test, public_predictions)

print("public test accuracy:", public_accuracy)
print("public test confusion matrix:")
print(public_cm)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=public_cm,
    display_labels=["negative", "positive"]
).plot()
plt.title("public test confusion matrix")
plt.show()

print(classification_report(
    y_public_test,
    public_predictions,
    target_names=["negative", "positive"],
    digits=4
))

## evaluation tokens not shown during training

This vectorizer is fitted using the training reviews, if a word were to appear in a public test review but did not show up in training, the CountVectorizer just doesn't create a feature for that new word.

This prevents any of the test set vocabulary information from leaking into training and vice versa.

## public test results achieved

The final model achieved 0.6750 accuracy on the public test file

### confusion matrix below

| | predicted negative | predicted positive |
|---|---:|---:|
| actual negative | 100 | 100 |
| actual positive | 30 | 170 |

## saving and reloading the model checkpoint

The complete pipeline is saved in `model_checkpoint/sentiment_model.joblib`. This contains the fitted vocabulary and the fitted logistic regression classifier.

In [ ]:
os.makedirs("model_checkpoint", exist_ok=True)

joblib.dump(final_model, "model_checkpoint/sentiment_model.joblib")

loaded_model = joblib.load("model_checkpoint/sentiment_model.joblib")
reloaded_predictions = loaded_model.predict(X_public_test)

print("Reloaded predictions identical:",
      np.array_equal(public_predictions, reloaded_predictions))

## creating the  public_test_predictions.csv file

the created file follows the structure, (id,predicted label), with the predicted label being either 0 or 1.

In [ ]:
prediction_df = pd.DataFrame({
    "id": public_test_df["id"],
    "predicted_label": public_predictions.astype(int)
})

prediction_df.to_csv("public_test_predictions.csv", index=False)

print(prediction_df.head())
print("Columns:", prediction_df.columns.tolist())
print("Rows:", len(prediction_df))

## Use of AI 

I had not grasped the concept of reloading a model checkpoint, so I had OpenAI create the corresponding code in the cell with the following prompt:

'Create a model checkpoint file that will contain all the needed files to reload the trained model, explain why these files are needed for the model to be reloaded.'

Once I imported the code to this notebook, I ran it after changing certain variables to be in line with code that I had already written, and it created the model checkpoint folder successfully, containing the necessary file named sentiment_model.joblib. This file contained a complete scikit learn pipeline with 2 main parts, a CountVectorizer and LogisticRegression which acts as the sentiment classifier.